# NAMAA female voice v11 — production quality gate
Female voice reference from original public NAMAA demo; measured speed, 0.60s pauses, 0.90 tempo, rejects/regen clipped utterances instead of publishing them. Does not touch Project 79.

In [ ]:
exec("import os,subprocess\nfrom pathlib import Path\npy=\"/kaggle/working/namaa_venv_v6/bin/python\"\nassert Path(py).exists(),\"Existing verified V6 Python is missing\"\nenv=os.environ.copy()\nenv.pop(\"PYTHONPATH\",None)\nenv.pop(\"PYTHONHOME\",None)\nenv[\"PYTHONNOUSERSITE\"]=\"1\"\nenv[\"HF_HOME\"]=\"/kaggle/working/namaa_hf_cache\"\npath=Path(\"/kaggle/working/namaa_female_calm_v2.py\")\npath.write_text(\"import os,time,json,subprocess,hashlib\\nfrom pathlib import Path\\nimport torch,torchaudio\\nfrom huggingface_hub import hf_hub_download,snapshot_download\\nfrom safetensors.torch import load_file\\nfrom chatterbox.mtl_tts import ChatterboxMultilingualTTS\\nroot=Path(\\\"/kaggle/working/namaa_female_calm_v2\\\")\\nroot.mkdir(parents=True,exist_ok=True)\\nchunks=[\\\"تخيّلي إن فيه ملك، الناس لحد دلوقتي بتتكلم عن غناه.\\\",\\\"اسمه مانسا موسى، وكان حاكم مالي في غرب أفريقيا.\\\",\\\"ده من حوالي سبعمية سنة. أيامها كانت مالي دولة غنية.\\\",\\\"ومشهورة بالدهب والملح، والتجارة اللي بتعدّي فيها.\\\",\\\"بس اللي خلّى العالم كله يسمع عنه، كانت رحلة الحج.\\\",\\\"خرج من بلده بقافلة كبيرة، ومعاه هدايا كتير.\\\",\\\"ولما وصل القاهرة، الناس اتكلمت عن كرمه.\\\",\\\"وعن الدهب اللي وزّعه، لدرجة إن الحكاية فضلت تتروى.\\\",\\\"لكن خدوا بالكم. مش كل رقم عن ثروته نقدر نصدّقه.\\\",\\\"زمان مكنش فيه بنوك ولا تقارير مالية بالشكل ده.\\\",\\\"فصعب نحسب هو كان معاه كام بالظبط.\\\",\\\"والمهم كمان إن مالي كانت مركز للعلم والتجارة.\\\",\\\"وفي تمبكتو، انتشرت مراكز المعرفة والمدارس.\\\",\\\"يعني أثر مانسا موسى، مش كله فلوس ودهب.\\\",\\\"القيمة الحقيقية، كانت في اللي سابه بعده.\\\",\\\"يا ترى لو كنت مكانه، كنت هتعمل إيه بكل الثروة دي؟\\\"]\\nassert torch.cuda.is_available(),\\\"NO_GPU\\\"\\nprint(\\\"BEGIN_V11_FEMALE_CALM_QC\\\",torch.cuda.get_device_name(0),flush=True)\\nref=hf_hub_download(repo_id=\\\"omarelshehy/NAMAA-Egyptian-Voice\\\",repo_type=\\\"space\\\",revision=\\\"224ac11f27e97136523102e5f28d27dbcb30a34e\\\",filename=\\\"female_voice.wav\\\")\\nmodel=ChatterboxMultilingualTTS.from_pretrained(device=\\\"cuda\\\")\\nweights=snapshot_download(\\\"NAMAA-Space/NAMAA-Egyptian-TTS\\\",allow_patterns=[\\\"t3_mtl23ls_v2.safetensors\\\"])\\nmodel.t3.load_state_dict(load_file(str(Path(weights)/\\\"t3_mtl23ls_v2.safetensors\\\"),device=\\\"cpu\\\"),strict=True)\\nmodel.t3.to(\\\"cuda\\\").eval()\\nprint(\\\"FEMALE_VOICE_READY\\\",flush=True)\\nstart=time.monotonic()\\nparts=[]\\ndetails=[]\\nfor i,text in enumerate(chunks,1):\\n    p=root/(\\\"line_%02d.wav\\\"%i)\\n    minimum=max(2.1,len(text)/18)\\n    valid=False\\n    audio=None\\n    seconds=0\\n    if p.is_file() and p.stat().st_size>3000:\\n        audio,sr=torchaudio.load(str(p))\\n        seconds=audio.shape[-1]/sr\\n        valid=sr==model.sr and seconds>=minimum\\n        if valid:print(\\\"CACHED_VALID\\\",i,round(seconds,2),flush=True)\\n    if not valid:\\n        for attempt in range(1,5):\\n            seed=72000+i*23+attempt*97\\n            torch.manual_seed(seed)\\n            candidate=model.generate(text,language_id=\\\"ar\\\",audio_prompt_path=ref if (i==1 and attempt==1) else None,exaggeration=0.38,cfg_weight=0.38,temperature=0.67+0.04*(attempt-1),repetition_penalty=1.12)\\n            candidate=candidate.detach().cpu()\\n            sec=candidate.shape[-1]/model.sr\\n            print(\\\"CANDIDATE\\\",i,\\\"ATTEMPT\\\",attempt,\\\"SECONDS\\\",round(sec,2),\\\"MIN\\\",round(minimum,2),flush=True)\\n            if sec>=minimum and float(candidate.abs().mean())>0.0005:\\n                audio=candidate\\n                seconds=sec\\n                torchaudio.save(str(p),audio,model.sr)\\n                valid=True\\n                break\\n    if not valid:\\n        raise RuntimeError(\\\"QUALITY_GATE_FAILED: LINE \\\"+str(i)+\\\" \\\"+text+\\\" audio last_seconds=\\\"+str(seconds))\\n    parts.append(audio)\\n    details.append({\\\"line\\\":i,\\\"seconds\\\":round(seconds,2),\\\"text\\\":text})\\n    print(\\\"ACCEPTED\\\",i,\\\"SECONDS\\\",round(seconds,2),\\\"TOTAL_RAW\\\",round(sum(d[\\\"seconds\\\"] for d in details),1),flush=True)\\npause=torch.zeros((1,int(model.sr*0.60)))\\nall_audio=torch.cat([w for i,a in enumerate(parts) for w in ([a] if i==0 else [pause,a])],dim=-1)\\noriginal=root/\\\"raw_qc.wav\\\"\\ntorchaudio.save(str(original),all_audio,model.sr)\\nmp3=root/\\\"namaa_female_calm_v2.mp3\\\"\\nwav=root/\\\"namaa_female_calm_v2.wav\\\"\\nsubprocess.run([\\\"ffmpeg\\\",\\\"-y\\\",\\\"-hide_banner\\\",\\\"-loglevel\\\",\\\"error\\\",\\\"-i\\\",str(original),\\\"-af\\\",\\\"atempo=0.90,loudnorm=I=-17:TP=-2:LRA=10\\\",\\\"-ar\\\",\\\"24000\\\",\\\"-ac\\\",\\\"1\\\",str(wav)],check=True)\\nsubprocess.run([\\\"ffmpeg\\\",\\\"-y\\\",\\\"-hide_banner\\\",\\\"-loglevel\\\",\\\"error\\\",\\\"-i\\\",str(wav),\\\"-codec:a\\\",\\\"libmp3lame\\\",\\\"-qscale:a\\\",\\\"3\\\",str(mp3)],check=True)\\nresult,sr=torchaudio.load(str(wav))\\ntotal_duration=round(result.shape[-1]/sr,2)\\nassert 60<=total_duration<=125,(\\\"OUT_OF_RANGE_AUDIO_LENGTH\\\",total_duration)\\nassert mp3.stat().st_size>200000\\nprint(\\\"CALM_FEMALE_QC_SUCCESS\\\",\\\"SECS\\\",total_duration,\\\"BYTES\\\",mp3.stat().st_size,\\\"SEGMENTS\\\",len(details),\\\"ELAPSED\\\",round(time.monotonic()-start,1),flush=True)\\n(root/\\\"qc_report.json\\\").write_text(json.dumps({\\\"duration\\\":total_duration,\\\"reference\\\":ref,\\\"settings\\\":{\\\"exaggeration\\\":0.38,\\\"cfg_weight\\\":0.38,\\\"temperature_base\\\":0.67,\\\"atempo\\\":0.90,\\\"pause_seconds\\\":0.60},\\\"clips\\\":details},ensure_ascii=False,indent=2),encoding=\\\"utf-8\\\")\\nimport requests\\nwith mp3.open(\\\"rb\\\") as f:\\n    r=requests.post(\\\"https://tmpfiles.org/api/v1/upload\\\",files={\\\"file\\\":(mp3.name,f,\\\"audio/mpeg\\\")},timeout=90)\\nprint(\\\"UPLOAD_HTTP\\\",r.status_code,flush=True)\\nprint(\\\"UPLOAD_TEXT\\\",r.text[:500],flush=True)\\nr.raise_for_status()\\nu=r.json()[\\\"data\\\"][\\\"url\\\"]\\nprint(\\\"QC_FEMALE_DOWNLOAD_URL\\\",u.replace(\\\"tmpfiles.org/\\\",\\\"tmpfiles.org/dl/\\\"),flush=True)\\n\",encoding=\"utf-8\")\nprint(\"RUN_NAMAA_FEMALE_V11\",flush=True)\nsubprocess.run([py,\"-u\",str(path)],check=True,env=env,timeout=900)\n")